# SDPO rollout analysis — anatomy of the brevity collapse

Every training rollout is a full completion (`<think>…</think> …prose… ```code```…`) with verdict, reward,
token count, teacher-eligibility, feedback. This notebook studies **how and why the model collapses to
terse, wrong answers**, using `src/rollout_analysis.py`.

**Data** (in `runs/iteration-*/evaldata/`, pulled from the `sdpo-outputs` volume):
- `iter10_32k_rollouts.jsonl` — star set: **untruncated** (32k cap) rollouts, steps 0–12
- `iter09_train_rollouts.jsonl` — 20k-cap rollouts (truncation contrast)
- eval samples `*_samples.jsonl` + judged `sdpo_passk_*.json` (base vs trained capability)

**Questions:** (1) *What* shrinks — reasoning or code? (2) *Why* — does SDPO distill from shorter rollouts?
(3) Does the cap cause truncation-failures? (4) What reasoning did a lost problem drop?

In [ ]:
import sys; sys.path.insert(0, "../src")   # or run from repo root with PYTHONPATH=src
import matplotlib.pyplot as plt
import rollout_analysis as ra
ROOT = ".."

df32 = ra.load_rollouts(f"{ROOT}/runs/iteration-10/evaldata/iter10_32k_rollouts.jsonl")
print(f"32k rollouts: {len(df32)} rows, steps {df32.step.min()}-{df32.step.max()}, "
      f"{df32.problem_id.nunique()} problems")
print("verdicts:", dict(df32.verdict.value_counts()))
df32.head(3)

## 1. The collapse — completion length over training
Median + p10–p90 tokens per step: the collapse curve everything below explains.

In [ ]:
traj = ra.length_trajectory(df32)
ra.plot_length_trajectory(traj); plt.show()
traj.round(0)

## 2. What collapses — reasoning vs code  (the key plot)
Split each completion into `<think>` block vs final code block. Thinking shrinks while code holds →
the model *skips reasoning*, it isn't just being concise.

In [ ]:
tvc = ra.think_vs_code_trajectory(df32)
ra.plot_think_vs_code(tvc); plt.show()
print("thinking %.0fk -> %.0fk chars; code stays ~%.1fk"
      % (tvc.think_chars.iloc[0]/1000, tvc.think_chars.iloc[-1]/1000, tvc.code_chars.mean()/1000))
tvc.round(3)

## 3. Quality shift — verdict mix over training
AC gives way to WA (confident wrong answers) as reasoning shrinks.

In [ ]:
vm = ra.verdict_mix(df32)
ra.plot_verdict_mix(vm); plt.show()
vm.round(3)

## 4. The mechanism — are SDPO's teachers shorter?
SDPO self-distills **from** `teacher_eligible` (successful) rollouts. Teachers shorter than the failures =
the model is literally taught to be brief. The collapse engine.

In [ ]:
ra.teacher_length_bias(df32).round(0)

## 5. Truncation-failure — does hitting the cap cause failures?
Do NO_CODE/failing rollouts cluster at the token cap (thought until budget ran out, no code)?
`truncation_analysis` quantifies the capped fraction + its AC rate — the **cap-accelerant** hypothesis.
Compare 32k (cap 32768) vs 20k (cap 20480).

In [ ]:
print("32k: length by verdict (NO_CODE sits at the 32768 cap):")
display(ra.length_by_verdict(df32).round(0))
print("32k: capped fraction + pass rate by step:")
display(ra.truncation_analysis(df32, cap_tokens=32768).round(3))
df20 = ra.load_rollouts(f"{ROOT}/runs/iteration-09/evaldata/iter09_train_rollouts.jsonl")
print("20k run: capped fraction (much higher early -> more truncation-failures):")
display(ra.truncation_analysis(df20, cap_tokens=20480).round(3))

## 6. One problem, degrading
`problem_trajectory(pid)` — mean length, AC rate, think/code per step for a single problem.

In [ ]:
pid = int(df32.problem_id.value_counts().index[0])
pt = ra.problem_trajectory(df32, pid)
fig, a1 = plt.subplots(figsize=(8,4)); a2 = a1.twinx()
a1.plot(pt.step, pt.n_tokens, "o-", color="steelblue"); a1.set_ylabel("mean n_tokens", color="steelblue")
a2.plot(pt.step, pt.ac_rate, "s--", color="crimson"); a2.set_ylabel("AC rate", color="crimson"); a2.set_ylim(0,1)
a1.set_xlabel("step"); plt.title(f"loj-{pid}: reasoning vs solve-rate"); plt.show()
pt.round(3)

## 7. Qualitative — read what the reasoning *lost*
Find a problem base solves but the trained model fails, then read the base thinking vs a collapsed
completion's thinking. Where the story gets concrete: *the model stopped doing the step it needed.*

In [ ]:
base = ra.load_eval_samples(f"{ROOT}/runs/iteration-09/evaldata/sdpo_passk_iter09dose_base_samples.jsonl")
base = ra.attach_verdicts(base, f"{ROOT}/runs/iteration-09/evaldata/sdpo_passk_iter09dose_base.json")
trained = ra.load_eval_samples(f"{ROOT}/runs/iteration-10/evaldata/sdpo_passk_iter1032k_ckpt6_samples.jsonl")
trained = ra.attach_verdicts(trained, f"{ROOT}/reports/iteration-10/data/sdpo_passk_iter1032k_ckpt6.json")
contrast = ra.find_contrast_examples(base, trained)
contrast.head(8)

In [ ]:
pid = int(contrast.iloc[0].id)
b = base[(base.id==pid) & (base.verdict=="AC")].iloc[0]
t = trained[trained.id==pid].sort_values("n_chars").iloc[0]
def peek(row, n=1200):
    tc = ra.split_think_code(row.completion)
    print(f"  verdict={row.verdict} n_chars={row.n_chars} think={tc['think_chars']} code={tc['code_chars']}")
    print("  THINKING (first chars):\n", row.completion[:n], "\n  ...")
print(f"===== loj-{pid} BASE (solves) ====="); peek(b)
print(f"\n===== loj-{pid} TRAINED 32k ckpt-6 (fails) ====="); peek(t)

## Summary panel (regenerates `reports/iteration-10/figures/rollout_analysis_demo.png`)

In [ ]:
fig, ax = plt.subplots(2,2, figsize=(14,9)); tb = ra.teacher_length_bias(df32)
ra.plot_length_trajectory(ra.length_trajectory(df32), ax[0,0])
ra.plot_think_vs_code(tvc, ax[0,1]); ra.plot_verdict_mix(vm, ax[1,0])
ax[1,1].plot(tb.step, tb["teacher=True"]/1000, "o-", color="darkorange", label="teacher-eligible")
ax[1,1].plot(tb.step, tb["teacher=False"]/1000, "s-", color="gray", label="not teacher")
ax[1,1].set_title("teachers vs non-teachers (k tokens)"); ax[1,1].legend(); ax[1,1].grid(alpha=.3)
fig.suptitle("SDPO collapse anatomy - 32k rollouts", fontweight="bold"); plt.tight_layout(rect=[0,0,1,.96]); plt.show()